# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [2]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from models_manager import ModelsManager
import gradio as gr
from IPython.display import Markdown, display

In [3]:
MODEL_GPT_4O_MINI = 'gpt-4o-mini'
MODEL_GPT_4_1_MINI = 'gpt-4.1-mini'
MODEL_GPT_5_NANO = 'gpt-5-nano'
CLIENT_OLLAMA = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    remote_client = OpenAI()
else:
    print("OpenAI API Key not set. You can't use remote models.")

local_client = CLIENT_OLLAMA

system_prompt = '''
You are a helpful assistant that can answer questions about the code.
By default, make sure to explain the answer in a way that is easy to understand.
If the user tells you that they are an advanced user or a professional, 
make sure to explain the answer in a way that is technical and detailed.
If the user tells you they are coming from a different programming language, or mentioned another programming language,
make sure to explain the answer in a way that is relevant to the mentioned language.
'''

In [ ]:
manager = ModelsManager(
    system_prompt=system_prompt, 
    remote_models=[MODEL_GPT_4O_MINI, MODEL_GPT_4_1_MINI, MODEL_GPT_5_NANO]
    )
models = manager.get_models()
choices = [(model.get('name'), model) for model in models.values()]

In [ ]:
def add_user_message(message, history):
    """Add user message to chat history and clear input."""
    return "", history + [{'role': 'user', 'content': message}]


def stream_bot_response(history, model):
    """Stream assistant response using full conversation history."""
    # Convert Gradio history to API format (list of {role, content})
    messages = [{'role': h['role'], 'content': h['content']} for h in history]
    history = history + [{'role': 'assistant', 'content': ''}]
    for chunk in manager.stream_chat_response(messages, model):
        history[-1]['content'] = chunk
        yield history



with gr.Blocks() as view:
    with gr.Row():
        model_selector = gr.Dropdown(choices=choices, label='Select a model', value=choices[0][1])
    chatbot = gr.Chatbot(height=600, type='messages')

    message_input = gr.Textbox(
        label='Your message:',
        placeholder='Ask a technical question... (Shift+Enter to send message, Enter for new line)',
        lines=3,
        elem_id='chat-input',
        scale=5
    )
    clear_btn = gr.ClearButton([message_input, chatbot])

    message_submit = message_input.submit(add_user_message, [message_input, chatbot], [message_input, chatbot], queue=False)
    message_submit.then(stream_bot_response, [chatbot, model_selector], chatbot)

view.launch(inbrowser=True)